In [14]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import random
import time

In [15]:
POPULATION_SIZE = 20
GENERATIONS = 10
MUTATION_RATE = 0.1
CROSSOVER_RATE = 0.8
ALPHA = 0.99  # Weight for F1 Score
BETA = 0.01   # Weight for Feature Reduction

# RBF SVM Settings (GPU)
BATCH_SIZE = 2048
EPOCHS_GA = 5           # Fast check for GA
EPOCHS_FINAL = 50       # Deep training for final result
LR = 0.001              # Lower LR for RBF stability
RFF_DIM = 1024          # Dimension of the RBF approximation
RBF_SIGMA = 2.0         # 'Gamma' equivalent (bandwidth)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Running on: {DEVICE}")

Running on: cuda


In [16]:
def load_and_preprocess(dataset_name):
    print(f"\n[{dataset_name}] Loading and Preprocessing...")
    
    if dataset_name == 'NSL-KDD':
        col_names = ["duration","protocol_type","service","flag","src_bytes",
            "dst_bytes","land","wrong_fragment","urgent","hot","num_failed_logins",
            "logged_in","num_compromised","root_shell","su_attempted","num_root",
            "num_file_creations","num_shells","num_access_files","num_outbound_cmds",
            "is_host_login","is_guest_login","count","srv_count","serror_rate",
            "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
            "diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count",
            "dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
            "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
            "dst_host_rerror_rate","dst_host_srv_rerror_rate","label", "difficulty"]
        
        train_df = pd.read_csv('KDDTrain+.txt', names=col_names)
        test_df = pd.read_csv('KDDTest+.txt', names=col_names)
        train_df.drop('difficulty', axis=1, inplace=True)
        test_df.drop('difficulty', axis=1, inplace=True)
        
        y_train = np.where(train_df['label'] == 'normal', 0, 1)
        y_test = np.where(test_df['label'] == 'normal', 0, 1)
        
        X_train = train_df.drop('label', axis=1)
        X_test = test_df.drop('label', axis=1)
        
        cat_cols = ["protocol_type", "service", "flag"]

    elif dataset_name == 'UNSW-NB15':
        train_df = pd.read_csv('UNSW_NB15_training-set.csv')
        test_df = pd.read_csv('UNSW_NB15_testing-set.csv')
        
        drop_cols = ['id', 'attack_cat']
        train_df.drop([c for c in drop_cols if c in train_df.columns], axis=1, inplace=True)
        test_df.drop([c for c in drop_cols if c in test_df.columns], axis=1, inplace=True)
        
        y_train = train_df['label'].values
        y_test = test_df['label'].values
        
        X_train = train_df.drop('label', axis=1)
        X_test = test_df.drop('label', axis=1)
        
        cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

    # One-Hot Encoding & Min-Max Scaling
    num_cols = [c for c in X_train.columns if c not in cat_cols]
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', MinMaxScaler(), num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
        ]
    )
    
    print("   Applying Transformations...")
    X_train_enc = preprocessor.fit_transform(X_train).astype(np.float32)
    X_test_enc = preprocessor.transform(X_test).astype(np.float32)
    
    return X_train_enc, y_train, X_test_enc, y_test

In [17]:
class RBFSVM(nn.Module):
    def __init__(self, input_dim, rff_dim=RFF_DIM, sigma=RBF_SIGMA):
        super(RBFSVM, self).__init__()
        # Random Fourier Features (Fixed weights, not trained)
        # This maps input to high-dim space where RBF kernel is linear
        self.rff_weights = nn.Parameter(torch.randn(input_dim, rff_dim) / sigma, requires_grad=False)
        self.rff_bias = nn.Parameter(torch.rand(rff_dim) * 2 * np.pi, requires_grad=False)
        
        # The learnable decision boundary
        self.fc = nn.Linear(rff_dim, 1)
        
    def forward(self, x):
        # Feature Mapping: cos(Wx + b)
        x_mapped = torch.cos(x @ self.rff_weights + self.rff_bias)
        return self.fc(x_mapped)

In [18]:
def train_evaluate_svm(X_train, y_train, X_val, y_val, feature_mask, epochs=5):
    # 1. Filter Features based on GA Mask
    indices = [i for i, x in enumerate(feature_mask) if x == 1]
    if len(indices) == 0: return 0.0 # No features selected
    
    # Subset Data
    X_tr_sub = X_train[:, indices]
    X_val_sub = X_val[:, indices]
    
    # To GPU
    X_tr_t = torch.tensor(X_tr_sub).to(DEVICE)
    y_tr_t = torch.tensor(np.where(y_train==0, -1, 1), dtype=torch.float32).to(DEVICE)
    X_val_t = torch.tensor(X_val_sub).to(DEVICE)
    
    # DataLoader
    dataset = TensorDataset(X_tr_t, y_tr_t)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    # 2. Initialize RBF SVM
    input_dim = len(indices)
    model = RBFSVM(input_dim).to(DEVICE)
    
    # Adam is better for RBF/Neural layers than SGD
    optimizer = optim.Adam(model.fc.parameters(), lr=LR)
    
    # 3. Training Loop
    model.train()
    for _ in range(epochs):
        for bx, by in loader:
            optimizer.zero_grad()
            out = model(bx).squeeze()
            # Hinge Loss
            loss = torch.mean(torch.clamp(1 - by * out, min=0))
            loss.backward()
            optimizer.step()
            
    # 4. Evaluation
    model.eval()
    with torch.no_grad():
        out = model(X_val_t).squeeze()
        preds = torch.where(out >= 0, 1, 0).cpu().numpy()
        
    return f1_score(y_val, preds, average='binary')

In [19]:
def fitness_function(chromosome, X_train, y_train, X_val, y_val):
    num_selected = sum(chromosome)
    total_features = len(chromosome)
    if num_selected == 0: return 0
    
    # Train RBF SVM on selected features
    f1 = train_evaluate_svm(X_train, y_train, X_val, y_val, chromosome, epochs=EPOCHS_GA)
    
    # Fitness = Accuracy vs Complexity tradeoff
    feature_ratio = num_selected / total_features
    fitness = (ALPHA * f1) + (BETA * (1.0 - feature_ratio))
    
    return fitness

In [20]:
def run_genetic_algorithm(X_train, y_train, X_test, y_test):
    num_features = X_train.shape[1]
    
    # Validation split for GA fitness check
    X_tr_sub, X_val, y_tr_sub, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
    
    # Init Population
    population = [[random.randint(0, 1) for _ in range(num_features)] for _ in range(POPULATION_SIZE)]
    
    print(f"   Starting GA with RBF SVM ({GENERATIONS} gens)...")
    
    for gen in range(GENERATIONS):
        scores = []
        for chrom in population:
            scores.append(fitness_function(chrom, X_tr_sub, y_tr_sub, X_val, y_val))
        
        best_score = max(scores)
        print(f"   Gen {gen+1}: Best Fitness = {best_score:.4f}")
        
        # Elitism & Tournament Selection
        new_population = []
        for _ in range(POPULATION_SIZE):
            parent1 = population[np.argmax(scores)]
            parent2 = population[random.randint(0, POPULATION_SIZE-1)]
            
            # Crossover
            if random.random() < CROSSOVER_RATE:
                point = random.randint(1, num_features-1)
                child = parent1[:point] + parent2[point:]
            else:
                child = parent1
            
            # Mutation
            for i in range(len(child)):
                if random.random() < MUTATION_RATE:
                    child[i] = 1 - child[i]
            
            new_population.append(child)
        population = new_population

    # Final Selection
    final_scores = [fitness_function(c, X_tr_sub, y_tr_sub, X_val, y_val) for c in population]
    best_chromosome = population[np.argmax(final_scores)]
    
    return best_chromosome

In [21]:
def process_dataset(name):
    X_train, y_train, X_test, y_test = load_and_preprocess(name)
    
    start_time = time.time()
    best_mask = run_genetic_algorithm(X_train, y_train, X_test, y_test)
    ga_time = time.time() - start_time
    
    selected_count = sum(best_mask)
    total_count = len(best_mask)
    print(f"\n[{name}] GA Completed in {ga_time:.2f}s")
    print(f"Selected {selected_count}/{total_count} features")

    # Final Training with More Epochs
    print("Training Final RBF SVM on Selected Features...")
    final_f1 = train_evaluate_svm(X_train, y_train, X_test, y_test, best_mask, epochs=EPOCHS_FINAL)
    
    print(f"Final F1 Score (RBF): {final_f1:.4f}")
    print("-" * 50)

In [ ]:
process_dataset('NSL-KDD')
process_dataset('UNSW-NB15')



[NSL-KDD] Loading and Preprocessing...
   Applying Transformations...
   Starting GA with RBF SVM (10 gens)...
